In [8]:
import sys
import os

sys.path.append(os.path.abspath("../.."))

from config.spark import spark


In [9]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,TimestampType

order_reviews_schema = StructType([
    StructField("review_id",StringType()),
    StructField("order_id",StringType()),
    StructField("review_score",StringType()),
    StructField("review_comment_title",StringType()),
    StructField("review_comment_message",TimestampType()),
    StructField("review_creation_date",TimestampType()),
    StructField("review_answer_timestamp",TimestampType()),
])


In [10]:
order_reviews_df = (
    spark.read.format('csv')
    .option('header','True')
  #  .option('inferschema','True')
  .schema(order_reviews_schema)
    .load('../../landing/files/olist_order_reviews_dataset.csv')
    
)

In [11]:
from pyspark.sql.functions import current_timestamp

order_reviews_df = order_reviews_df.withColumn("ingestion_time",current_timestamp())

In [12]:
order_reviews_df.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: timestamp (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)



In [13]:
order_reviews_df.write.mode("overwrite").saveAsTable("bronze.order_reviews")

In [14]:
spark.sql(
    """
    select * from bronze.order_reviews
    """
).show(100)

+--------------------+--------------------+-------------------+--------------------+----------------------+--------------------+-----------------------+--------------------+
|           review_id|            order_id|       review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|      ingestion_time|
+--------------------+--------------------+-------------------+--------------------+----------------------+--------------------+-----------------------+--------------------+
|8278360118b83106a...|0558d3449b449e4da...|                  4|                   7|                  NULL| 2018-07-11 00:00:00|    2018-07-14 15:22:36|2026-10-19 15:29:...|
|3a0fd124d55874a20...|825318ce17c91088b...|                  1|                NULL|                  NULL| 2018-06-26 00:00:00|    2018-06-27 21:46:52|2026-10-19 15:29:...|
|2deac296a2c2efa80...|ad66f64c10df4649e...|                  5|                NULL|                  NULL| 2017-07-20 00:00:00|  